<div
     style="padding: 20px;
            color: white;
            font-size: 250%;
            text-align: center;
            display: fill;
            border-radius: 5px;
            background-color: #023c66;
            overflow: hidden;
            font-weight: 700;
            border: 5px solid #F28C28;"
     >
    Lyrics Clustering Analysis
</div>

This notebook prepares song lyrics for clustering analysis. The workflow includes data cleaning, text preprocessing, feature engineering, selection of the number of clusters, and final clustering.

## **1. Data Preparation**
________________________________
The dataset is loaded, duplicate records are removed, lyric text is normalized, and observations with very little usable text are excluded.

In [ ]:
data_path = "spotify_song_lyrics-1.csv"

df = pd.read_csv(data_path, engine="python")
df = df[["artist", "song", "text"]].copy()

df["artist"] = df["artist"].astype(str).str.strip()
df["song"] = df["song"].astype(str).str.strip()
df["text"] = df["text"].astype(str).str.strip()
df["raw_word_count"] = df["text"].str.split().str.len()

df = df.drop_duplicates()
df = (
    df.sort_values(["artist", "song", "raw_word_count"], ascending=[True, True, False])
      .drop_duplicates(subset=["artist", "song"], keep="first")
      .reset_index(drop=True)
)

df = df[df["raw_word_count"] >= 40].copy()

stop_words = ENGLISH_STOP_WORDS.union({
    "verse", "chorus", "repeat", "repeats", "bridge", "intro", "outro",
    "hook", "feat", "ft", "yeah", "uh", "oh", "oooh", "oooo",
    "la", "na", "woah", "whoa", "hey", "ya", "gotta", "wanna", "gonna"
})

df["clean_text"] = df["text"].str.lower()
df["clean_text"] = df["clean_text"].str.replace(r"\[(.*?)\]", " ", regex=True)
df["clean_text"] = df["clean_text"].str.replace(r"\((.*?)\)", " ", regex=True)
df["clean_text"] = df["clean_text"].str.replace(r"[^a-z\s']", " ", regex=True)
df["clean_text"] = df["clean_text"].str.replace(r"\b\w\b", " ", regex=True)
df["clean_text"] = df["clean_text"].str.replace(r"\s+", " ", regex=True).str.strip()

df["clean_word_count"] = df["clean_text"].str.split().str.len()
df = df[df["clean_word_count"] >= 20].reset_index(drop=True)

df.shape

: 

In [ ]:
summary = pd.Series({
    "rows": len(df),
    "unique_artists": df["artist"].nunique(),
    "unique_artist_song_pairs": df[["artist", "song"]].drop_duplicates().shape[0],
    "median_raw_words": df["raw_word_count"].median(),
    "median_clean_words": df["clean_word_count"].median()
})
summary

## **2. Feature Engineering**

1.   List item
2.   List item


________________________________
Lyrics are converted into numerical representations using count-based and TF-IDF vectorization. TF-IDF is then reduced to a dense lower-dimensional representation with truncated singular value decomposition.

count_vectorizer = CountVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.5,
    max_features=10000
)

tfidf_vectorizer = TfidfVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.5,
    max_features=10000,
    sublinear_tf=True
)

X_count = count_vectorizer.fit_transform(df["clean_text"])
X_tfidf = tfidf_vectorizer.fit_transform(df["clean_text"])

svd = TruncatedSVD(n_components=75, random_state=42)
X_embed = svd.fit_transform(X_tfidf)
X_embed = Normalizer(copy=False).fit_transform(X_embed)

pd.Series({
    "count_matrix_rows": X_count.shape[0],
    "count_matrix_columns": X_count.shape[1],
    "tfidf_matrix_rows": X_tfidf.shape[0],
    "tfidf_matrix_columns": X_tfidf.shape[1],
    "embedding_dimensions": X_embed.shape[1],
    "explained_variance": round(svd.explained_variance_ratio_.sum(), 3)
})

In [ ]:
top_terms = pd.DataFrame({
    "term": count_vectorizer.get_feature_names_out(),
    "count": np.asarray(X_count.sum(axis=0)).ravel()
}).sort_values("count", ascending=False)

top_terms.head(15)

## **3. Cluster Selection**
________________________________
Several candidate values of k are evaluated. The number of clusters is selected primarily using silhouette score, with inertia included as a supporting measure.

In [ ]:
candidate_ks = range(4, 11)
sample_size = min(6000, len(X_embed))
sample_idx = np.random.default_rng(42).choice(len(X_embed), size=sample_size, replace=False)
X_sample = X_embed[sample_idx]

scores = []
for k in candidate_ks:
    model = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=2048, n_init=10)
    labels = model.fit_predict(X_embed)
    scores.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X_sample, labels[sample_idx])
    })

cluster_scores = pd.DataFrame(scores)
cluster_scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(cluster_scores["k"], cluster_scores["inertia"], marker="o")
axes[0].set_xlabel("Number of clusters")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Plot")

axes[1].plot(cluster_scores["k"], cluster_scores["silhouette"], marker="o")
axes[1].set_xlabel("Number of clusters")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette Plot")

plt.tight_layout()
plt.show()

In [ ]:
best_k = int(cluster_scores.sort_values(["silhouette", "k"], ascending=[False, True]).iloc[0]["k"])
best_k

The selected value of k is the one with the strongest silhouette score. If multiple values are similar, the smaller value is preferred because it is easier to interpret.

## **4. Final Clustering**
________________________________
The final clustering model is estimated using the selected number of clusters. Cluster sizes and representative terms are then reviewed to summarize the resulting groups.

In [ ]:
kmeans = MiniBatchKMeans(n_clusters=best_k, random_state=42, batch_size=2048, n_init=20)
df["cluster"] = kmeans.fit_predict(X_embed)

df["cluster"].value_counts().sort_index()

In [ ]:
terms = np.array(tfidf_vectorizer.get_feature_names_out())
top_n_terms = 5
cluster_summary = []

for cluster_id in sorted(df["cluster"].unique()):
    cluster_mask = (df["cluster"] == cluster_id).to_numpy()
    cluster_size = int(cluster_mask.sum())
    mean_tfidf = np.asarray(X_tfidf[cluster_mask].mean(axis=0)).ravel()
    top_terms = terms[mean_tfidf.argsort()[::-1][:top_n_terms]]

    cluster_summary.append({
        "cluster": cluster_id,
        "size": cluster_size,
        "top_terms": ", ".join(top_terms)
    })

pd.DataFrame(cluster_summary).sort_values("size", ascending=False)